In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names
print("Feature names:", feature_names)
print("Target names:", target_names)
print("Shape of X:", X.shape)
print("Counts of each class:", np.bincount(y))

Feature names: ['mean radius' 'mean texture' 'mean perimeter' 'mean area'
 'mean smoothness' 'mean compactness' 'mean concavity'
 'mean concave points' 'mean symmetry' 'mean fractal dimension'
 'radius error' 'texture error' 'perimeter error' 'area error'
 'smoothness error' 'compactness error' 'concavity error'
 'concave points error' 'symmetry error' 'fractal dimension error'
 'worst radius' 'worst texture' 'worst perimeter' 'worst area'
 'worst smoothness' 'worst compactness' 'worst concavity'
 'worst concave points' 'worst symmetry' 'worst fractal dimension']
Target names: ['malignant' 'benign']
Shape of X: (569, 30)
Counts of each class: [212 357]


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.3, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
svc = SVC(kernel='rbf', probability=True, random_state=42)
svc.fit(X_train_scaled, y_train)
y_pred = svc.predict(X_test_scaled)
y_proba = svc.predict_proba(X_test_scaled)[:, 1]
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))

Confusion Matrix:
[[ 62   2]
 [  2 105]]

Classification Report:
              precision    recall  f1-score   support

   malignant       0.97      0.97      0.97        64
      benign       0.98      0.98      0.98       107

    accuracy                           0.98       171
   macro avg       0.98      0.98      0.98       171
weighted avg       0.98      0.98      0.98       171

Accuracy: 0.9766081871345029
ROC AUC: 0.9978095794392523


In [ ]:
param_grid = {
'C': [0.1, 1, 10, 100],
'gamma': ['scale', 0.001, 0.01, 0.1, 1],
'kernel': ['rbf', 'linear']
}
grid = GridSearchCV(SVC(probability=True), param_grid, cv=5,
scoring='accuracy', n_jobs=-1)
grid.fit(X_train_scaled, y_train)
print("Best parameters:", grid.best_params_)
print("Best cross-val score:", grid.best_score_)

Best parameters: {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'}
Best cross-val score: 0.9799050632911392


In [ ]:
best_svc = grid.best_estimator_
y_pred_best = best_svc.predict(X_test_scaled)
y_proba_best = best_svc.predict_proba(X_test_scaled)[:, 1]
print("\n=== Tuned SVC Performance ===")
print("Confusion Matrix (tuned):")
print(confusion_matrix(y_test, y_pred_best))
print("\nClassification Report (tuned):")
print(classification_report(y_test, y_pred_best,
target_names=target_names))
print("Accuracy (tuned):", accuracy_score(y_test, y_pred_best))
print("ROC AUC (tuned):", roc_auc_score(y_test, y_proba_best))


=== Tuned SVC Performance ===
Confusion Matrix (tuned):
[[ 60   4]
 [  1 106]]

Classification Report (tuned):
              precision    recall  f1-score   support

   malignant       0.98      0.94      0.96        64
      benign       0.96      0.99      0.98       107

    accuracy                           0.97       171
   macro avg       0.97      0.96      0.97       171
weighted avg       0.97      0.97      0.97       171

Accuracy (tuned): 0.9707602339181286
ROC AUC (tuned): 0.9956191588785046
